In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))



In [10]:
from music_tracker.models import Track, Artist, Playlist

ImportError: cannot import name 'DatabaseManager' from 'music_tracker.db' (c:\Users\t490s\26k-zindua\python-projects\music-playlist-tracker\music_tracker\db.py)

In [2]:
from music_tracker.models import Track, Artist, Playlist
# ── Create some Track objects ─────────────────────────────────────────────────
t1 = Track("Blinding Lights", "The Weeknd", duration=200, listeners=5_200_000, 
           tags=["pop", "synth-pop"], playcount=100_000_000)
t2 = Track("HUMBLE.", "Kendrick Lamar", duration=177, listeners=4_800_000,
           tags=["hip-hop", "rap"], playcount=80_000_000)
t3 = Track("Bohemian Rhapsody", "Queen", duration=354, listeners=6_100_000,
           tags=["rock", "classic rock"], playcount=200_000_000)
t4 = Track("Blinding Lights", "The Weeknd", duration=200)  # duplicate

print(t1)
print(t2)
print(t3)
print(f"\nDuration type  : {t1.classify_length()}")
print(f"Is popular?    : {t1.is_popular()}")
print(f"As tuple       : {t1.to_tuple()}")

🎵 Blinding Lights — The Weeknd (3:20)
🎵 HUMBLE. — Kendrick Lamar (2:57)
🎵 Bohemian Rhapsody — Queen (5:54)

Duration type  : mid
Is popular?    : True
As tuple       : ('Blinding Lights', 'The Weeknd', 200, 5200000, 100000000, 'pop,synth-pop', '')


In [10]:
import sys, os
from dotenv import load_dotenv

# .env is in the same folder the notebook runs from
load_dotenv(os.path.join(os.getcwd(), '.env'))

key = os.getenv("LASTFM_API_KEY")
print(f"Key found: {key}")

Key found: 57a8a20624b8df13662b894681495491


In [11]:

track_record = ("Blinding Lights", "The Weeknd", 200)

title, artist, duration = track_record      # unpacking
print(f"Title    : {title}")
print(f"Artist   : {artist}")
print(f"Duration : {duration}s")

raw_top_tracks = get_artist_top_tracks("Kendrick Lamar", limit=5)

track_tuples = []
for t in raw_top_tracks:
    record = (
        clean_name(t["name"]),
        "Kendrick Lamar",
        int(t.get("listeners", 0))
    )
    track_tuples.append(record)

print("\n Track tuples (immutable records):")
for record in track_tuples:
    print(f"  {record}")

Title    : Blinding Lights
Artist   : The Weeknd
Duration : 200s
❌ API error: 400 Client Error: Bad Request for url: http://ws.audioscrobbler.com/2.0/?method=artist.getTopTracks&artist=Kendrick+Lamar&limit=5&format=json

 Track tuples (immutable records):


In [8]:

tracks_to_tag = [
    ("Blinding Lights", "The Weeknd"),
    ("Save Your Tears", "The Weeknd"),
    ("HUMBLE.", "Kendrick Lamar"),
    ("DNA.", "Kendrick Lamar"),
]

all_tags = set()        
weeknd_tags = set()
kendrick_tags = set()

for title, artist in tracks_to_tag:
    tags = get_track_tags(title, artist)
    all_tags.update(tags)
    if artist == "The Weeknd":
        weeknd_tags.update(tags)
    else:
        kendrick_tags.update(tags)

print(f"  The Weeknd tags    : {weeknd_tags}")
print(f"  Kendrick tags      : {kendrick_tags}")
print(f"  All unique tags    : {all_tags}")


shared = weeknd_tags & kendrick_tags        
unique_to_weeknd = weeknd_tags - kendrick_tags   
print(f"\n Shared tags        : {shared}")
print(f" Only The Weeknd    : {unique_to_weeknd}")

❌ API error: 400 Client Error: Bad Request for url: http://ws.audioscrobbler.com/2.0/?method=track.getTopTags&track=Blinding+Lights&artist=The+Weeknd&format=json
❌ API error: 400 Client Error: Bad Request for url: http://ws.audioscrobbler.com/2.0/?method=track.getTopTags&track=Save+Your+Tears&artist=The+Weeknd&format=json
❌ API error: 400 Client Error: Bad Request for url: http://ws.audioscrobbler.com/2.0/?method=track.getTopTags&track=HUMBLE.&artist=Kendrick+Lamar&format=json
❌ API error: 400 Client Error: Bad Request for url: http://ws.audioscrobbler.com/2.0/?method=track.getTopTags&track=DNA.&artist=Kendrick+Lamar&format=json
  The Weeknd tags    : set()
  Kendrick tags      : set()
  All unique tags    : set()

 Shared tags        : set()
 Only The Weeknd    : set()


In [6]:


artists = ["The Weeknd", "Kendrick Lamar", "Adele", "Arctic Monkeys"]

from music_tracker.api import get_artist_info

print("🎤 Artist Tier Classification:\n")
for name in artists:
    data = get_artist_info(name)
    if not data:
        print(f"   {name} — could not fetch data")
        continue

    listeners = int(data["stats"]["listeners"])

    
    if listeners >= 10_000_000:
        tier = " Global Superstar"
    elif listeners >= 5_000_000:
        tier = " Mainstream"
    elif listeners >= 1_000_000:
        tier = " Rising"
    else:
        tier = " Underground"

    print(f"  {name:<25} {listeners:>12,} listeners → {tier}")

🎤 Artist Tier Classification:

❌ API error: 400 Client Error: Bad Request for url: http://ws.audioscrobbler.com/2.0/?method=artist.getInfo&artist=The+Weeknd&format=json
   The Weeknd — could not fetch data
❌ API error: 400 Client Error: Bad Request for url: http://ws.audioscrobbler.com/2.0/?method=artist.getInfo&artist=Kendrick+Lamar&format=json
   Kendrick Lamar — could not fetch data
❌ API error: 400 Client Error: Bad Request for url: http://ws.audioscrobbler.com/2.0/?method=artist.getInfo&artist=Adele&format=json
   Adele — could not fetch data
❌ API error: 400 Client Error: Bad Request for url: http://ws.audioscrobbler.com/2.0/?method=artist.getInfo&artist=Arctic+Monkeys&format=json
   Arctic Monkeys — could not fetch data


In [ ]:

print(" Building Track objects from API:\n")
top = get_artist_top_tracks("The Weeknd", limit=5)
tracks = []

for i, t in enumerate(top, start=1):
    info = get_track_info(t["name"], "The Weeknd")
    if not info:
        continue

    duration_ms = int(info.get("duration", 0))
    duration_sec = duration_ms // 1000
    tags = get_track_tags(t["name"], "The Weeknd")

    track = Track(
        title=clean_name(info["name"]),
        artist_name="The Weeknd",
        duration=duration_sec,
        listeners=int(info.get("listeners", 0)),
        playcount=int(info.get("playcount", 0)),
        tags=tags,
        url=info.get("url", "")
    )
    tracks.append(track)
    print(f"  {i}. {track}")


print("\n While loop — add tracks until playlist has 3:\n")
playlist = Playlist("The Weeknd Essentials", "Top tracks from The Weeknd")
index = 0

while len(playlist) < 3 and index < len(tracks):
    playlist.add_track(tracks[index])
    index += 1

print(playlist.summary())

In [ ]:


artists_to_compare = ["The Weeknd", "Kendrick Lamar"]
artist_tag_sets = {}      

for artist_name in artists_to_compare:
    top = get_artist_top_tracks(artist_name, limit=3)
    tag_set = set()

    for t in top:
        tags = get_track_tags(t["name"], artist_name)
        tag_set.update(tags)

    artist_tag_sets[artist_name] = tag_set

    
    summary_tuple = (artist_name, len(top), len(tag_set))
    print(f" Summary tuple: {summary_tuple}")

a1_tags = artist_tag_sets["The Weeknd"]
a2_tags = artist_tag_sets["Kendrick Lamar"]

overlap = a1_tags & a2_tags
print(f"\n Genre overlap between artists: {overlap if overlap else 'None'}")
print(f" Unique to The Weeknd        : {a1_tags - a2_tags}")
print(f" Unique to Kendrick Lamar    : {a2_tags - a1_tags}")

In [9]:
import os
print("Notebook is running from:", os.getcwd())
print("Files here:", os.listdir(os.getcwd()))

Notebook is running from: c:\Users\t490s\26k-zindua\python-projects\music-playlist-tracker
Files here: ['.env', '.git', '.gitignore', 'env', 'music_tracker', 'notebook.ipynb', 'README.md', 'requirements.txt', 'tests']
